# 9. Experimental compact WATERec workflow

This separate workflow uses **multistage distillation**, not the one-step Florence recipe or the standard R/RS training command. It is opt-in and does not bundle the measured checkpoint. The one-epoch dummy defaults demonstrate software behavior, **not benchmark accuracy**.

Read [the workflow guide](../docs/compact_workflow.md) first. A supplied standard teacher's training/pretraining provenance and absence of validation/test leakage are your responsibility. The native charset is limited; unsupported text is rejected. Automatic resume is not implemented: use fresh output/deployment folders.

Install the training/ONNX/notebook extras first. TensorRT additionally needs a compatible target, CUDA toolkit/NVCC and a C compiler. The standard notebooks and R/RS model options remain unchanged.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import yaml
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "waterec_ocr").is_dir())
CONFIG = ROOT / "configs/experimental/compact_dummy_price.yaml"
config = yaml.safe_load(CONFIG.read_text())
def config_path(value):
    p = Path(value).expanduser()
    return p.resolve() if p.is_absolute() else (CONFIG.parent / p).resolve()
DATA = config_path(config["dataset_root"])
OUTPUT = config_path(config["output_dir"])
CHECKPOINT = OUTPUT / "model.pt"
DEPLOYMENT = OUTPUT / "deployment"
IMAGE = next(iter(sorted((DATA / "val").glob("*.png"))))
def run(*args):
    return subprocess.run([sys.executable, "-m", "waterec_ocr.waterec_compact_cli", *map(str, args)], cwd=ROOT, check=True)
RUN_TRAINING = False
RUN_ONNX = False
RUN_TENSORRT = False
RUN_BENCHMARK = False
print("Config paths resolve relative to", CONFIG.parent)
print("Native configuration:", {k: config[k] for k in ["task", "tokens", "recovery_epochs", "scorer_epochs", "joint_epochs", "restart_epochs"]})


## Train a fresh run

`train_compact` optionally fine-tunes a standard teacher using FIT only, initializes 24 MLP factors by SVD, recovers rank weights without pruning, freezes that rank parent as a full-memory response/attention teacher, warms the image scorer and jointly trains token selection. An optional restart gets fresh Adam moments. GT characters plus EOS set the token denominator; image KL uses the image count. No boxes are used.

The default dummy budget is 64, so up to 64 patches are retained; it does **not** test the 2048-to-512 compression operating point. `configs/experimental/compact_price.yaml` is a separate starting profile, also not an accuracy recipe. Configure epochs and evaluate independently.

If the output already contains a run, choose a new `output_dir`; do not delete evidence to pretend to resume.

In [ ]:
if RUN_TRAINING:
    run("train", "--config", CONFIG)
else:
    print("Training disabled; set RUN_TRAINING=True for the fictional demo.")


## Inspect typed checkpoints and run native inference

Stage `best.pt`/`last.pt` retain architecture, optimizer, RNG, teacher/source/input hashes and history. `model.pt` is the selected typed inference model. Validation selects earliest raw exact then mean normalized Levenshtein similarity, including epoch zero. Without validation the terminal is fixed. Validation targets never train the teacher.

Native inference takes images only. A standard R/RS checkpoint cannot be renamed into a compact one.

In [ ]:
if CHECKPOINT.is_file():
    completion = json.loads((OUTPUT / "completion.json").read_text())
    print("Stage selections:", [(s["name"], s["selected_epoch"]) for s in completion["stages"]])
    run("infer", "--backend", "torch", "--device", "cpu", "--checkpoint", CHECKPOINT, "--image", IMAGE)
    run("evaluate", "--backend", "torch", "--device", "cpu", "--checkpoint", CHECKPOINT,
        "--folder", DATA / "val", "--output", OUTPUT / "native_evaluation.json")
else:
    print("No checkpoint yet; training has not been run.")


## Export and test ONNX on CPU

The exported graph has a fixed input-token budget and batch 1 by default. Unicode additions use this B1 path; do not enable the separate ASCII fixed-capacity path below. New native training resolutions require a new export/profile; existing engines do not become dynamic-resolution or dynamic-batch models. Export alone does not establish accuracy at another resolution.


In [ ]:
if RUN_ONNX:
    if not CHECKPOINT.is_file():
        raise FileNotFoundError("Run training first or provide your typed compact checkpoint.")
    run("export", "--checkpoint", CHECKPOINT, "--deployment", DEPLOYMENT)
    run("infer", "--backend", "onnx", "--deployment", DEPLOYMENT,
        "--checkpoint", CHECKPOINT, "--image", IMAGE)
else:
    print("ONNX export/inference disabled.")


## Target-specific TensorRT build and inference (optional)

Only enable this on a compatible NVIDIA/CUDA target. Rebuild and verify outputs on your actual target; the qualified RTX 5090/static-B1 2048 profile does not qualify every bucket or edge device. The budget-64 dummy workflow and actual CLI export/build/infer/evaluate chain passed functional CPU/GPU checks on fictional examples. It does not inherit the trained 2048 profile’s accuracy or resource measurements. Compact TensorRT configured budgets are capped at 2048; not every budget up to that cap is accuracy-qualified. No orientation/speculative/unchecked runtime option is enabled here.

In [ ]:
if RUN_TENSORRT:
    run("build", "--deployment", DEPLOYMENT)
    run("infer", "--backend", "tensorrt", "--deployment", DEPLOYMENT,
        "--checkpoint", CHECKPOINT, "--image", IMAGE)
    run("evaluate", "--backend", "tensorrt", "--deployment", DEPLOYMENT,
        "--checkpoint", CHECKPOINT, "--folder", DATA / "val", "--output", OUTPUT / "trt_evaluation.json")
else:
    print("TensorRT build/inference disabled; no GPU work performed.")


## Benchmark your model separately

Predecoded RGB and encoded-file timing have different boundaries. Use separate fresh processes and record batch size, parameters, input profile, dataset, accuracy and memory together. Process GPU usage includes context; host RSS may include decoded-image caches and libraries. Training peak allocation is not inference memory.

The separately trained audited price checkpoint has 17,913,953 parameters, a 2048-patch input and 512 retained patches after six blocks. Public runtime v3 measured **1.356083 ms RGB / 1.620985 ms encoded, 708 MiB** on RTX 5090/static batch 1, preserving all 206 native strings. Its **161/206 raw (78.16%)**, **187/206 numeric (90.78%)**, **132/144 AI-reviewed numeric** and **90.18% mean normalized Levenshtein similarity** are distinct metrics. AI references are uncertified, and that checkpoint's results do not transfer to demonstration weights. See [the research record](../docs/compact_research.md).

In [ ]:
if RUN_BENCHMARK:
    run("benchmark", "--backend", "tensorrt", "--deployment", DEPLOYMENT,
        "--checkpoint", CHECKPOINT, "--folder", DATA / "val", "--repeats", "3", "--predecoded",
        "--output", OUTPUT / "benchmark_RGB.json")
    run("benchmark", "--backend", "tensorrt", "--deployment", DEPLOYMENT,
        "--checkpoint", CHECKPOINT, "--folder", DATA / "val", "--repeats", "3",
        "--output", OUTPUT / "benchmark_encoded.json")
else:
    print("Benchmark disabled; no latency or memory result is implied.")


## Optional ASCII fixed-capacity batches

This separate export requires a 2048-budget ASCII compact checkpoint with 512
retained tokens. The default fictional budget-64 run above intentionally does
not qualify. Set `RUN_FIXED_BATCH=True` only after supplying a matching model.
Fixed capacities 1/2/4/8 process true batched rows; partial tails duplicate the
first real row and discard dummy outputs. Report batch latency separately from
real-image throughput. Extended Unicode checkpoints must omit `--static-batch`
and use the original B1 path; Unicode batching is explicitly unsupported.


In [ ]:
RUN_FIXED_BATCH = False
STATIC_BATCH = 2  # 1, 2, 4 or 8; separate graph/profile, not dynamic batching
BATCH_DEPLOYMENT = OUTPUT / f"deployment_batch{STATIC_BATCH}"
if RUN_FIXED_BATCH:
    if config["tokens"] != 2048 or config.get("additional_characters"):
        raise ValueError("Fixed-batch deployment requires an ASCII 2048→512 checkpoint.")
    run("export", "--checkpoint", CHECKPOINT, "--deployment", BATCH_DEPLOYMENT,
        "--static-batch", STATIC_BATCH)
    run("infer", "--backend", "onnx", "--deployment", BATCH_DEPLOYMENT,
        "--folder", DATA / "val", "--batch-size", STATIC_BATCH)
    if RUN_TENSORRT:
        run("build", "--deployment", BATCH_DEPLOYMENT)
        run("evaluate", "--backend", "tensorrt", "--deployment", BATCH_DEPLOYMENT,
            "--folder", DATA / "val", "--batch-size", STATIC_BATCH)
else:
    print("Fixed-batch export disabled; original B1 path unchanged.")
